# Model 002: Recommender Path (from predictors to recommendations)

This notebook is a roadmap for turning the current review outcome models into a true recommendation system.

Current models predict:
- `recommended`
- `is_helpful`
- `votes_helpful`

These are useful, but they do not directly recommend *new games* yet.

Goal: user enters a review, system returns games they are likely to enjoy.

## Why current setup is not enough

Current targets answer:
- "Will a review be positive/helpful?"

A recommender must answer:
- "What should this user try next?"

So we need recommendation-specific candidate generation and ranking.

## System design: 2-stage recommender

1. **Candidate generation**
   - Produce a shortlist of potential games.
   - Content-based first (fastest MVP): review-text similarity to game profiles.

2. **Ranking/scoring**
   - Re-rank candidates with a blended score.
   - Combine similarity with predicted `recommended` / helpfulness signals.

## Phase 1 (MVP): Content-based recommender

### Inputs
- User free-text review (optionally simple metadata like playtime)

### Build artifacts
- `game_profile.parquet` or equivalent:
  - one row per game (`app_id`, `app_name`)
  - profile text (aggregate positive reviews)
  - profile embedding/vector
- `vectorizer.pkl` or embedding model reference

### Online inference flow
1. Vectorize user review.
2. Compute similarity to game profile vectors.
3. Return top-N games by similarity.

This gives a true recommendation output quickly.

## Phase 2: Hybrid ranking (use current models)

Use existing models as ranking features for candidate games.

Example blended rank score:

`rank_score = 0.6 * text_similarity + 0.3 * p_recommended + 0.1 * p_is_helpful`

Potential features to rank with:
- text similarity
- `p(recommended)`
- `p(is_helpful)` or expected `votes_helpful`
- game popularity/volume priors

Then tune weights on validation data.

## Evaluation shifts for recommendation

Keep prediction metrics, but add recommendation metrics as primary:
- Precision@K
- Recall@K
- MAP@K
- NDCG@K

Secondary diagnostics:
- coverage/diversity of recommendations
- popularity bias checks
- per-game and per-segment performance

## Concrete next tasks (execution order)

1. Build game text profiles from training reviews.
2. Implement similarity retrieval and return top-10 games.
3. Add a simple API endpoint for recommendations.
4. Add hybrid scoring with `recommended` / helpfulness model outputs.
5. Evaluate with @K metrics and compare against similarity-only baseline.
6. Iterate on ranking weights/features and document findings.

In [ ]:
# Optional: quick pseudocode sketch for MVP retrieval

# user_text -> vectorize -> cosine similarity with game profile vectors -> top_k
#
# user_vec = vectorizer.transform([user_text])
# sims = cosine_similarity(user_vec, game_profile_matrix).ravel()
# top_idx = np.argsort(-sims)[:10]
# recommendations = game_profiles.iloc[top_idx][["app_id", "app_name"]]
# recommendations